In [6]:
import csv
import time

def genera_csv_ecommerce_scenario1_massivo(nome_file, numero_totale_nodi=1000000):
    print(f"⚙️ Inizio generazione dataset massivo E-commerce: {nome_file}")
    print(f"📊 Nodi totali (Prodotti del catalogo): {numero_totale_nodi}")
    
    inizio = time.time()
    
    # Apriamo il file in modalità scrittura ('w')
    with open(nome_file, mode='w', newline='') as file_csv:
        writer = csv.writer(file_csv)
        
        # Scriviamo l'intestazione
        writer.writerow(['Source', 'Target'])
        
        # I nodi 0, 1, 2 sono i Best-Seller (non hanno archi in uscita, non raccomandano nessuno)
        # I Prodotti Isolati partono dal nodo 3 fino a 999.999
        # Per evitare di stampare 1 milione di righe singole (lento), le prepariamo in blocchi (chunking)
        
        blocco_dati = []
        dimensione_blocco = 50000 # Scrive sul file ogni 50.000 prodotti isolati elaborati
        
        # Ogni prodotto isolato genera un acquisto diretto verso i 3 Best-Seller
        # I prodotti isolati non fanno cross-selling tra di loro, quindi non hanno altri archi in uscita
        for prodotto in range(3, numero_totale_nodi):
            # Il prodotto isolato punta esclusivamente ai 3 Best-Seller
            blocco_dati.append([prodotto, 0])
            blocco_dati.append([prodotto, 1])
            blocco_dati.append([prodotto, 2])
            
            # Quando il blocco è pieno, lo scriviamo sul disco e svuotiamo la memoria RAM
            if len(blocco_dati) >= dimensione_blocco * 3:
                writer.writerows(blocco_dati)
                blocco_dati.clear()
                
        # Scriviamo gli eventuali dati rimasti nel blocco alla fine del ciclo
        if blocco_dati:
            writer.writerows(blocco_dati)
            
    fine = time.time()
    print(f"✅ Generazione completata con successo in {fine - inizio:.2f} secondi!")
    print(f"Il file '{nome_file}' è pronto nella cartella specificata.")

# ==========================================
# ESECUZIONE
# ==========================================
N_NODI = 1000000

# Ho aggiornato il nome del file e il probabile percorso per differenziarlo dal Caso Studio 1
NOME_FILE = '../Caso di Studio 2/DataSet_CasoStudio2/Rete_1M/dataset_scenario1_1MILIONE_cs2.csv'

genera_csv_ecommerce_scenario1_massivo(NOME_FILE, N_NODI)

⚙️ Inizio generazione dataset massivo E-commerce: ../Caso di Studio 2/DataSet_CasoStudio2/Rete_1M/dataset_scenario1_1MILIONE_cs2.csv
📊 Nodi totali (Prodotti del catalogo): 1000000
✅ Generazione completata con successo in 0.80 secondi!
Il file '../Caso di Studio 2/DataSet_CasoStudio2/Rete_1M/dataset_scenario1_1MILIONE_cs2.csv' è pronto nella cartella specificata.


In [4]:
import csv
import random
import time

def genera_csv_ecommerce_scenario2_massivo(nome_file, n_nodi=1000000):
    print(f"⚙️ Inizio generazione dataset massivo E-Commerce (Scenario 2): {nome_file}")
    inizio = time.time()

    # Inizializziamo il seed a 42. Questo è CRUCIALE: garantisce che la generazione 
    # casuale estragga gli STESSI IDENTICI numeri generati nello Scenario 1.
    random.seed(42)

    with open(nome_file, mode='w', newline='') as file_csv:
        writer = csv.writer(file_csv)
        writer.writerow(['Source', 'Target'])

        blocco_dati = []
        dimensione_blocco = 100000 # Scrittura su disco ogni 100.000 righe per svuotare la RAM

        def scrivi_blocco():
            """Funzione helper per svuotare la RAM scrivendo i dati sul CSV."""
            nonlocal blocco_dati
            if blocco_dati:
                writer.writerows(blocco_dati)
                blocco_dati.clear()

        # ==========================================
        # PASSO 1: BEST-SELLER (0, 1, 2)
        # ==========================================
        # Bundle/Circolo chiuso: 0->1, 1->2, 2->0
        # Questi prodotti si rimandano a vicenda, senza suggerire nient'altro nel catalogo
        blocco_dati.extend([[0, 1], [1, 2], [2, 0]])

        # ==========================================
        # PASSO 2: PRODOTTI ISOLATI (3 -> 499.999)
        # ==========================================
        print("📊 Generazione dei Prodotti Isolati (nessun cross-selling) in corso...")
        for prodotto_isolato in range(3, 500000):
            # Generano acquisti diretti solo per i 3 Best-Seller
            blocco_dati.extend([[prodotto_isolato, 0], [prodotto_isolato, 1], [prodotto_isolato, 2]])
            
            # Svuota la RAM se il blocco è pieno
            if len(blocco_dati) >= dimensione_blocco:
                scrivi_blocco()

        # ==========================================
        # PASSO 3: PRODOTTI CORRELATI E HUB (500.000 -> 999.999)
        # ==========================================
        print("🌐 Generazione dei Prodotti Correlati e Hub in corso...")
        start_organici = 500000
        end_organici = 999999

        for prodotto_correlato in range(start_organici, end_organici + 1):
            # 1. Tutti i prodotti correlati raccomandano i 3 Best-Seller
            blocco_dati.extend([[prodotto_correlato, 0], [prodotto_correlato, 1], [prodotto_correlato, 2]])

            # 2. Logica Hub (Prodotti civetta/base)
            # Eleggiamo a "Hub" un prodotto ogni 500 (circa 1000 Hub totali nel catalogo)
            is_hub = (prodotto_correlato % 500 == 0)
            
            # Gli Hub generano 25 raccomandazioni (cross-selling), i prodotti normali solo 2
            num_raccomandazioni = 25 if is_hub else 2

            # Generazione delle raccomandazioni (cross-selling) all'interno di questa fascia
            for _ in range(num_raccomandazioni):
                target_casuale = random.randint(start_organici, end_organici) 
                
                # Evita l'auto-raccomandazione (un prodotto non raccomanda sé stesso)
                if target_casuale != prodotto_correlato: 
                    blocco_dati.append([prodotto_correlato, target_casuale])

            if len(blocco_dati) >= dimensione_blocco:
                scrivi_blocco()

        # Svuota gli ultimi dati rimasti in memoria alla fine del ciclo
        scrivi_blocco()

    fine = time.time()
    print(f"✅ Generazione completata con successo in {fine - inizio:.2f} secondi!")
    print(f"Il file '{nome_file}' è pronto.")

# ==========================================
# ESECUZIONE
# ==========================================
NOME_FILE = '../Caso di Studio 2/DataSet_CasoStudio2/Rete_1M/dataset_scenario2_1MILIONE_cs2.csv'

genera_csv_ecommerce_scenario2_massivo(NOME_FILE)

⚙️ Inizio generazione dataset massivo E-Commerce (Scenario 2): ../Caso di Studio 2/DataSet_CasoStudio2/Rete_1M/dataset_scenario2_1MILIONE_cs2.csv
📊 Generazione dei Prodotti Isolati (nessun cross-selling) in corso...
🌐 Generazione dei Prodotti Correlati e Hub in corso...
✅ Generazione completata con successo in 1.27 secondi!
Il file '../Caso di Studio 2/DataSet_CasoStudio2/Rete_1M/dataset_scenario2_1MILIONE_cs2.csv' è pronto.


In [5]:
import csv
import random
import time

def genera_csv_ecommerce_scenario3_massivo(nome_file, n_nodi=1000000):
    print(f"⚙️ Inizio generazione dataset massivo E-commerce (Scenario 3 Organico): {nome_file}")
    inizio = time.time()
    
    # Seed per garantire la riproducibilità scientifica assoluta tra i vari esperimenti
    random.seed(42)

    with open(nome_file, mode='w', newline='') as file_csv:
        writer = csv.writer(file_csv)
        writer.writerow(['Source', 'Target'])

        # Ottimizzazione della scrittura su disco
        blocco_dati = []
        dimensione_blocco = 100000 # Salva su disco e svuota la RAM ogni 100k archi

        def scrivi_blocco():
            nonlocal blocco_dati
            if blocco_dati:
                writer.writerows(blocco_dati)
                blocco_dati.clear()

        # Definizione dei Best-Seller (0, 1, 2)
        best_sellers = [0, 1, 2] 
        
        # ===================================================
        # DEFINIZIONE DEI MICRO-HUB MERCEOLOGICI
        # ===================================================
        # Selezioniamo un cluster di nodi strategici (multipli di 1000).
        # Questi articoli agiranno come leader di categoria / prodotti civetta.
        micro_hubs = set(range(1000, n_nodi, 1000)) 
        micro_hubs_list = list(micro_hubs)
        
        # ===============================================
        # PASSO 1: SCAMBIO BIDIREZIONALE (Best-Seller -> Micro-Hub)
        # ==============================================
        print("🎯 Generazione raccomandazioni inverse (Best-Seller -> Micro-Hub strategici)...")
        for bs in best_sellers:
            # Ogni Best-Seller raccomanda un set di 150 articoli scelti ESCLUSIVAMENTE tra i Micro-Hub
            hub_selezionati = random.sample(micro_hubs_list, 150)
            for hub in hub_selezionati:
                blocco_dati.append([bs, hub]) # Aggiunge la raccomandazione dal Best-Seller all'Hub

        # ===============================================
        # PASSO 2: RETE CATALOGO E DINAMICHE CROSS-SELLING
        # ==============================================
        print("🌐 Generazione cross-selling organico (Cluster, Co-acquisti e Bridging)...")
        
        # Iteriamo su tutti i prodotti del catalogo (da 3 a 999.999)
        for prodotto in range(3, n_nodi):
            
            # 1. Preferential Attachment Asimmetrico (Tutti puntano ai top)
            if random.random() < 0.90: blocco_dati.append([prodotto, 0]) 
            if random.random() < 0.70: blocco_dati.append([prodotto, 1]) 
            if random.random() < 0.50: blocco_dati.append([prodotto, 2]) 
            
            # 2. Generazione dell'Attività Locale (Cluster Merceologici)
            if prodotto in micro_hubs:
                # Questo nodo è un hub. Raccomanda densamente i prodotti vicini (stessa categoria).
                for j in range(1, 15):
                    if prodotto + j < n_nodi:
                        blocco_dati.append([prodotto, prodotto + j]) 
            else:
                # Prodotto normale. Raccomanda solo l'articolo adiacente.
                if prodotto + 1 < n_nodi:
                    blocco_dati.append([prodotto, prodotto + 1]) 
                
            # 3. Chiusure Triadiche (Co-acquisti / Bought Together)
            # Simula matematicamente il legame tra articoli acquistati di frequente
            if prodotto % 3 == 0 and (prodotto - 2) >= 3:
                blocco_dati.append([prodotto, prodotto - 2])
                
            # 4. Bridging mirato (Suggerimenti cross-categoria)
            # Il 5% dei prodotti genera una raccomandazione fuori dal proprio cluster locale
            if random.random() < 0.05:
                # L'80% di queste raccomandazioni viene attratto dai Micro-Hub 
                if random.random() < 0.80:
                    target_casuale = random.choice(micro_hubs_list)
                else:
                    target_casuale = random.randint(3, n_nodi - 1) 
                
                # Evita l'auto-raccomandazione
                if target_casuale != prodotto: 
                    blocco_dati.append([prodotto, target_casuale]) 
            
            # Salvataggio chunk
            if len(blocco_dati) >= dimensione_blocco:
                scrivi_blocco()

        scrivi_blocco()

    fine = time.time()
    print(f"✅ Generazione Scenario Massivo E-commerce completata con successo in {fine - inizio:.2f} secondi!")

# Esecuzione del file
NOME_FILE = '../Caso di Studio 2/DataSet_CasoStudio2/Rete_1M/dataset_scenario3_1MILIONE_cs2.csv'
genera_csv_ecommerce_scenario3_massivo(NOME_FILE)

⚙️ Inizio generazione dataset massivo E-commerce (Scenario 3 Organico): ../Caso di Studio 2/DataSet_CasoStudio2/Rete_1M/dataset_scenario3_1MILIONE_cs2.csv
🎯 Generazione raccomandazioni inverse (Best-Seller -> Micro-Hub strategici)...
🌐 Generazione cross-selling organico (Cluster, Co-acquisti e Bridging)...
✅ Generazione Scenario Massivo E-commerce completata con successo in 1.18 secondi!
